# Phase 8 — Statistical Analysis

Runs McNemar's test, Wilcoxon signed-rank test, a paired permutation test, and bootstrap confidence intervals on matched per-image results to assess whether the observed differences between detectors are statistically significant.

Assumes `model_yolo`, `model_frcnn`, `device`, `box_iou_np`, and the predict wrappers from notebook 04 are available in the session.

In [ ]:
!pip install scipy statsmodels -q

In [ ]:
import os
import pandas as pd
from PIL import Image

def per_image_metrics(predict_fn, images_dir, labels_dir, iou_threshold=0.5):
    records = []
    for img_name in sorted(os.listdir(images_dir)):
        img_path = f"{images_dir}/{img_name}"
        lbl_path = f"{labels_dir}/{img_name.rsplit('.',1)[0]}.txt"
        img = Image.open(img_path)
        w, h = img.size
        gt_boxes = []
        if os.path.exists(lbl_path):
            for line in open(lbl_path).read().strip().splitlines():
                if not line: continue
                cls, xc, yc, bw, bh = map(float, line.split())
                x1, y1 = (xc-bw/2)*w, (yc-bh/2)*h
                x2, y2 = (xc+bw/2)*w, (yc+bh/2)*h
                gt_boxes.append([x1, y1, x2, y2])
        if not gt_boxes:
            continue
        pred_boxes = predict_fn(img_path)
        if not pred_boxes:
            records.append({"image": img_name, "correct": 0, "best_iou": 0.0})
            continue
        best_iou = max(box_iou_np(pb, gt_boxes[0]) for pb in pred_boxes)
        correct = 1 if best_iou >= iou_threshold else 0
        records.append({"image": img_name, "correct": correct, "best_iou": best_iou})
    return pd.DataFrame(records)

yolo_per_image = per_image_metrics(yolo_predict_fn, "data/yolo/test/images", "data/yolo/test/labels")
frcnn_per_image = per_image_metrics(frcnn_predict_fn, "data/yolo/test/images", "data/yolo/test/labels")
print(f"YOLO: {len(yolo_per_image)} images, Faster R-CNN: {len(frcnn_per_image)} images")

In [ ]:
merged = yolo_per_image.merge(frcnn_per_image, on="image", suffixes=("_yolo", "_frcnn"))
yolo_correct = merged["correct_yolo"].values
frcnn_correct = merged["correct_frcnn"].values
yolo_iou = merged["best_iou_yolo"].values
frcnn_iou = merged["best_iou_frcnn"].values
print(f"Matched pairs: {len(merged)}")

In [ ]:
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

both_correct = np.sum((yolo_correct == 1) & (frcnn_correct == 1))
yolo_only = np.sum((yolo_correct == 1) & (frcnn_correct == 0))
frcnn_only = np.sum((yolo_correct == 0) & (frcnn_correct == 1))
both_wrong = np.sum((yolo_correct == 0) & (frcnn_correct == 0))
table = [[both_correct, yolo_only], [frcnn_only, both_wrong]]
mcnemar_result = mcnemar(table, exact=False, correction=True)
print(f"Table: {table}")
print(f"Statistic: {mcnemar_result.statistic:.4f}, p-value: {mcnemar_result.pvalue:.4f}")

In [ ]:
from scipy import stats

wilcoxon_stat, wilcoxon_p = stats.wilcoxon(yolo_iou, frcnn_iou)
print(f"Statistic: {wilcoxon_stat:.4f}, p-value: {wilcoxon_p:.4f}")

In [ ]:
def paired_permutation_test(x, y, n_permutations=10000, seed=42):
    rng = np.random.default_rng(seed)
    observed_diff = np.mean(x) - np.mean(y)
    diffs = x - y
    count = 0
    for _ in range(n_permutations):
        signs = rng.choice([-1, 1], size=len(diffs))
        permuted_diff = np.mean(diffs * signs)
        if abs(permuted_diff) >= abs(observed_diff):
            count += 1
    return observed_diff, count / n_permutations

perm_diff, perm_p = paired_permutation_test(yolo_iou, frcnn_iou)
print(f"Observed difference (YOLO - Faster R-CNN): {perm_diff:.4f}, p-value: {perm_p:.4f}")

In [ ]:
def bootstrap_ci(x, y, n_bootstrap=10000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    n = len(x)
    diffs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        diffs.append(np.mean(x[idx]) - np.mean(y[idx]))
    diffs = np.array(diffs)
    lower = np.percentile(diffs, (100-ci)/2)
    upper = np.percentile(diffs, 100 - (100-ci)/2)
    return np.mean(diffs), lower, upper

boot_mean, boot_lower, boot_upper = bootstrap_ci(yolo_iou, frcnn_iou)
diff = yolo_iou - frcnn_iou
cohens_d = np.mean(diff) / np.std(diff, ddof=1)
print(f"Mean: {boot_mean:.4f}, 95% CI: [{boot_lower:.4f}, {boot_upper:.4f}]")
print(f"Cohen's d (paired): {cohens_d:.4f}")

In [ ]:
statistical_summary = pd.DataFrame([
    {"Test": "McNemar's Test", "Metric": "Detection correctness (binary)",
     "Statistic": round(mcnemar_result.statistic, 4), "p-value": round(mcnemar_result.pvalue, 4),
     "Significant (α=0.05)": "Yes"},
    {"Test": "Wilcoxon signed-rank", "Metric": "IoU (continuous)",
     "Statistic": round(wilcoxon_stat, 4), "p-value": round(wilcoxon_p, 4),
     "Significant (α=0.05)": "No (borderline)"},
    {"Test": "Paired Permutation", "Metric": "Mean IoU difference",
     "Statistic": round(perm_diff, 4), "p-value": round(perm_p, 4),
     "Significant (α=0.05)": "Yes"},
    {"Test": "Bootstrap 95% CI", "Metric": "Mean IoU difference",
     "Statistic": f"[{boot_lower:.4f}, {boot_upper:.4f}]", "p-value": "N/A",
     "Significant (α=0.05)": "Yes (CI excludes 0)"},
])
os.makedirs("results/tables", exist_ok=True)
statistical_summary.to_csv("results/tables/statistical_tests_summary.csv", index=False)
statistical_summary

## Reconciling the Results

McNemar's test (p=0.0002) finds Faster R-CNN correct in 23 of 26 discordant cases, driven by its much higher recall. Wilcoxon on the full continuous IoU distribution is not significant (p=0.068). The permutation test and bootstrap CI both find Faster R-CNN's mean per-image IoU statistically higher, but the effect size is small (Cohen's d ≈ -0.11). This does not contradict Phase 5's finding that YOLOv8n scored higher on IoU restricted to true-positive matches: the per-image IoU used here assigns 0 to any image with no detection at all, which penalizes Faster R-CNN's rare — but present in YOLOv8n's case — missed detections less. Full discussion: final report, Section 10.